In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
sensor_path = "dbfs:/FileStore/iot_krish/raw/sensor_readings.json"
sensor_df = spark.read.option("multiLine", True).option("inferSchema", True).json(sensor_path)
sensor_df.printSchema()
display(sensor_df)

In [0]:
sensor_df = spark.read.option("inferSchema", True).json(sensor_path)

In [0]:
sensor_bronze_df = (
    sensor_df
    .withColumn("_ingested_at", current_timestamp())
    .withColumn("_source_file", lit("sensor_readings.json"))
)

In [0]:
(sensor_bronze_df.write
 .format("delta")
 .mode("overwrite")
 .saveAsTable("iot_krish.bronze.sensor_readings"))

In [0]:
devices_path = "dbfs:/FileStore/iot_krish/raw/devices.json"

devices_df = spark.read.json(devices_path)

(devices_df.write
 .format("delta")
 .mode("overwrite")
 .saveAsTable("iot_krish.bronze.devices"))

In [0]:
locations_path = "dbfs:/FileStore/iot_krish/raw/locations.json"

locations_df = spark.read.json(locations_path)

(locations_df.write
 .format("delta")
 .mode("overwrite")
 .saveAsTable("iot_krish.bronze.locations"))

In [0]:
%sql
ALTER TABLE iot_krish.bronze.sensor_readings
DROP CONSTRAINT IF EXISTS valid_temperature; ALTER TABLE iot_krish.bronze.sensor_readings ADD CONSTRAINT valid_temperature
CHECK (temperature_c >= -50 AND temperature_c <= 200);

In [0]:
bad_df = sensor_bronze_df.withColumn("temperature_c", col("temperature_c").cast("string"))

try:
    (bad_df.write
     .format("delta")
     .mode("append")
     .saveAsTable("iot_krish.bronze.sensor_readings"))
except Exception as e:
    print("Expected schema enforcement error:")
    print(e)

In [0]:
"""sensor_batch2_path = "dbfs:/FileStore/iot_krish/raw/sensor_readings_batch2.json"

checkpoint_path = "dbfs:/FileStore/iot_krish/checkpoints"

sensor_stream_df = (
    spark.readStream
    .format("cloudFiles")
    .option("cloudFiles.schemaLocation", "dbfs:/FileStore/iot_krish/schema")
    .option("cloudFiles.format", "json")
    .load(sensor_batch2_path)
)

(
    sensor_stream_df.writeStream
    .format("delta")
    .option("checkpointLocation", checkpoint_path)
    .trigger(availableNow=True)
    .toTable("iot_krish.bronze.sensor_readings_stream")
)"""